# Sprint 2 — Ingeniería de Targets y Split Temporal
## Proyecto: Productividad Asesores de Negocios

---

### Objetivo del sprint

Construir dos formulaciones del target en paralelo y tomar la decisión formal
de cuál usar como target principal, basada en evidencia cuantitativa.

| Target | Nivel | Variable | Pregunta de negocio |
|--------|-------|----------|---------------------|
| **Target_A** | Semanal | `DIAS_MORA_RANGO` con split temporal | ¿Cómo estará la cartera la próxima semana? |
| **Target_B** | Asesor | TDNC en cuartiles | ¿Este asesor es de alto o bajo riesgo estructural? |

### Advertencia metodológica

> El split temporal en Target_A debe hacerse por fecha de corte (`HOY`), no por
> índice de fila. Un split aleatorio permitiría que semanas futuras informen
> el entrenamiento — exactamente el leakage que corregimos de la tesis.


## 1. Configuración

In [1]:
# Librerías
import warnings
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

DB_PATH    = Path('../data/advisor_risk.duckdb')
INTERIM    = Path('../data/interim')
FIG_DIR    = Path('../reports/figures')
TABLE_NAME = 'tabla_hist_asesores'

INTERIM.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect(str(DB_PATH), read_only=True)
df   = conn.execute(f'SELECT * FROM {TABLE_NAME}').df()
conn.close()

for col in ['HOY', 'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f'Dataset cargado: {len(df):,} filas')
print(f'Periodo: {df["HOY"].min().date()} -> {df["HOY"].max().date()}')
print(f'Asesores unicos: {df["CODIGO_ASESOR"].nunique():,}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset cargado: 1,218,831 filas
Periodo: 2016-06-22 -> 2019-01-30
Asesores unicos: 2,004


## 2. Target_A — Split temporal sobre DIAS_MORA_RANGO

El split se hace por **fecha de corte (`HOY`)**, no por índice de fila.

- **Train:** todos los registros con `HOY <= fecha_corte`
- **Test:** todos los registros con `HOY > fecha_corte`

La fecha de corte se elige en el percentil 80 de las fechas únicas de `HOY`.


In [2]:
def build_target_a(df, pct_train=0.80):
    '''
    Split temporal sobre DIAS_MORA_RANGO.
    El corte se hace por fecha (HOY), no por fila.
    Garantiza que max(train) < min(test).
    '''
    EXCLUIR = [
        'PASE1', 'PASE15', 'PASE30',
        'DIAS_MORA_RANGO', 'DIAS_MORA',
        'ASESOR', 'NOMBRE', 'COORDINADOR',
        'NUMERO_EMPLEADO', 'N_SUCURSAL', 'HOY',
    ]
    TARGET_COL = 'DIAS_MORA_RANGO'

    # Fecha de corte: percentil pct_train de fechas unicas
    fechas_unicas = sorted(df['HOY'].dropna().unique())
    idx_corte     = int(len(fechas_unicas) * pct_train) - 1
    fecha_corte   = fechas_unicas[idx_corte]

    n_train_sem = idx_corte + 1
    n_test_sem  = len(fechas_unicas) - n_train_sem

    print(f'  Total cortes temporales : {len(fechas_unicas)}')
    print(f'  Cortes en train         : {n_train_sem} (hasta {pd.Timestamp(fecha_corte).date()})')
    print(f'  Cortes en test          : {n_test_sem} (desde {pd.Timestamp(fechas_unicas[idx_corte+1]).date()})')

    mask_train = df['HOY'] <= fecha_corte
    mask_test  = df['HOY'] >  fecha_corte

    df_train = df[mask_train].copy()
    df_test  = df[mask_test].copy()

    # Verificacion anti-leakage
    max_train = df_train['HOY'].max()
    min_test  = df_test['HOY'].min()
    assert max_train < min_test, 'ERROR: solapamiento temporal entre train y test'
    print(f'  Anti-leakage OK: max(train)={max_train.date()} < min(test)={min_test.date()}')

    feature_cols = [c for c in df.columns if c not in EXCLUIR]
    X_train = df_train[feature_cols]
    X_test  = df_test[feature_cols]
    y_train = df_train[TARGET_COL]
    y_test  = df_test[TARGET_COL]

    return X_train, X_test, y_train, y_test, fecha_corte, feature_cols


print('=== TARGET_A — Split Temporal ===')
X_train_A, X_test_A, y_train_A, y_test_A, fecha_corte_A, features_A = build_target_a(df)

print(f'  Train: {len(X_train_A):,} filas ({len(X_train_A)/len(df)*100:.1f}%)')
print(f'  Test : {len(X_test_A):,} filas ({len(X_test_A)/len(df)*100:.1f}%)')
print(f'  Features disponibles: {len(features_A)}')


=== TARGET_A — Split Temporal ===
  Total cortes temporales : 137
  Cortes en train         : 109 (hasta 2018-07-18)
  Cortes en test          : 28 (desde 2018-07-25)
  Anti-leakage OK: max(train)=2018-07-18 < min(test)=2018-07-25
  Train: 993,911 filas (81.5%)
  Test : 224,920 filas (18.5%)
  Features disponibles: 36


In [3]:
def distribucion_clases(y, nombre):
    dist = y.value_counts(dropna=False).reset_index()
    dist.columns = ['clase', 'frecuencia']
    dist['porcentaje'] = (dist['frecuencia'] / len(y) * 100).round(2)
    dist['conjunto']   = nombre
    return dist

dist_train_A = distribucion_clases(y_train_A, 'Train')
dist_test_A  = distribucion_clases(y_test_A,  'Test')

print('TRAIN:')
display(dist_train_A[['clase','frecuencia','porcentaje']])
print('TEST:')
display(dist_test_A[['clase','frecuencia','porcentaje']])

# Verificar distributional shift
merged = dist_train_A.merge(dist_test_A, on='clase', suffixes=('_train','_test'), how='outer').fillna(0)
merged['diff_pct'] = (merged['porcentaje_train'] - merged['porcentaje_test']).abs().round(2)
merged = merged.sort_values('diff_pct', ascending=False)
print('\n=== DISTRIBUTIONAL SHIFT ===')
print(merged[['clase','porcentaje_train','porcentaje_test','diff_pct']].to_string(index=False))
max_shift = merged['diff_pct'].max()
if max_shift > 5:
    print(f'\nShift temporal detectado: diferencia maxima = {max_shift:.1f}pp')
else:
    print(f'\nSin shift significativo: diferencia maxima = {max_shift:.1f}pp')


TRAIN:


,clase,frecuencia,porcentaje
0,CARTERA_A_TIEMPO,695433,69.9700
1,MORA_MAYOR_90,171791,17.2800
2,MORA_61_90,30023,3.0200
3,MORA_31_60,29190,2.9400
4,MORA_16_30,19600,1.9700
5,MORA_1_3,19114,1.9200
6,MORA_8_15,14423,1.4500
7,MORA_4_7,14337,1.4400


TEST:


,clase,frecuencia,porcentaje
0,CARTERA_A_TIEMPO,137773,61.2500
1,MORA_MAYOR_90,57780,25.6900
2,MORA_61_90,8128,3.6100
3,MORA_31_60,7630,3.3900
4,MORA_16_30,4962,2.2100
5,MORA_8_15,3114,1.3800
6,MORA_4_7,2785,1.2400
7,MORA_1_3,2748,1.2200



=== DISTRIBUTIONAL SHIFT ===
           clase  porcentaje_train  porcentaje_test  diff_pct
CARTERA_A_TIEMPO           69.9700          61.2500    8.7200
   MORA_MAYOR_90           17.2800          25.6900    8.4100
        MORA_1_3            1.9200           1.2200    0.7000
      MORA_61_90            3.0200           3.6100    0.5900
      MORA_31_60            2.9400           3.3900    0.4500
      MORA_16_30            1.9700           2.2100    0.2400
        MORA_4_7            1.4400           1.2400    0.2000
       MORA_8_15            1.4500           1.3800    0.0700

Shift temporal detectado: diferencia maxima = 8.7pp


## 3. Target_B — Tasa de Deterioro Neto de Cartera (TDNC) por asesor

La TDNC mide cuánto se deterioró la cartera de un asesor en relación a su tamaño promedio:

**TDNC = mean(ATRASO) / mean(CARTERA_HOY)**

Los cuartiles Q1–Q4 clasifican a los asesores de menor a mayor riesgo.

El dataset pasa de 1,218,831 filas (nivel semanal) a ~2,004 filas (nivel asesor).

> Con ~2,004 observaciones se usará **k-fold estratificado (k=5)** en lugar de
> hold-out simple. Si la varianza del macro F1 entre folds supera 0.10, el modelo
> se considera inestable.


In [4]:
def build_target_b(df):
    '''
    Colapsa el dataset a nivel asesor y construye cuartiles de TDNC.
    TDNC = mean(ATRASO) / mean(CARTERA_HOY)
    Target: Q1_BAJO / Q2_MEDIO_BAJO / Q3_MEDIO_ALTO / Q4_ALTO
    '''
    print('Colapsando dataset a nivel asesor...')

    AGG_MEDIA = [
        'GRUPOS', 'CLIENTES', 'PRESTAMO', 'CLIENTES_PRESTAMO',
        'CARTERA_HOY', 'CARTERA_SEMANT', 'INCREMENTO_CARTERA',
        'ATRASO', 'TASA_PROM', 'PROVISION_HOY', 'PROVISION_SEMANT',
        'GASTO_PROVISION_SEMANAL', 'CLIENTES_NUEVOS',
        'DESEMBOLSO_CLIENTES_NUEVOS', 'DIAS_MORA',
    ]
    AGG_ULTIMA = [
        'REGION', 'SUCURSAL', 'CODIGO_SUCURSAL', 'COORDINADOR',
        'PUESTO', 'PRODUCTO', 'AREA', 'SEXO', 'ESTADO_CIVIL',
        'EXP_MICROFINANZAS', 'NIVEL_ESTUDIOS', 'EDAD',
        'RANGO_CICLO', 'DIA_PAGO', 'TIPO_BAJA', 'MOTIVO_BAJA',
        'FECHA_ALTA', 'FECHA_BAJA', 'FECHA_NACIMIENTO',
    ]

    agg_media   = df.groupby('CODIGO_ASESOR')[AGG_MEDIA].mean()
    agg_ultima  = df.groupby('CODIGO_ASESOR')[AGG_ULTIMA].last()
    agg_semanas = df.groupby('CODIGO_ASESOR')['HOY'].nunique().rename('N_SEMANAS_OBS')
    agg_p_fecha = df.groupby('CODIGO_ASESOR')['HOY'].min().rename('PRIMERA_OBS')
    agg_u_fecha = df.groupby('CODIGO_ASESOR')['HOY'].max().rename('ULTIMA_OBS')

    df_asesor = pd.concat([
        agg_media, agg_ultima, agg_semanas, agg_p_fecha, agg_u_fecha
    ], axis=1).reset_index()

    # Calcular TDNC
    epsilon = 1e-6
    df_asesor['TDNC'] = df_asesor['ATRASO'] / (df_asesor['CARTERA_HOY'] + epsilon)
    df_asesor['TDNC'] = df_asesor['TDNC'].clip(lower=0)

    # Asignar cuartiles
    df_asesor['TARGET_B'] = pd.qcut(
        df_asesor['TDNC'],
        q=4,
        labels=['Q1_BAJO', 'Q2_MEDIO_BAJO', 'Q3_MEDIO_ALTO', 'Q4_ALTO'],
        duplicates='drop',
    )

    print(f'Dataset colapsado: {len(df_asesor):,} asesores')
    return df_asesor


df_asesores = build_target_b(df)

print('\n=== DISTRIBUCIÓN DE TARGET_B ===')
dist_B = df_asesores['TARGET_B'].value_counts(dropna=False).reset_index()
dist_B.columns = ['cuartil', 'asesores']
dist_B['porcentaje'] = (dist_B['asesores'] / len(df_asesores) * 100).round(2)
display(dist_B)

print('\n=== TDNC POR CUARTIL ===')
display(df_asesores.groupby('TARGET_B')['TDNC'].describe().round(4))


Colapsando dataset a nivel asesor...
Dataset colapsado: 2,004 asesores

=== DISTRIBUCIÓN DE TARGET_B ===


,cuartil,asesores,porcentaje
0,Q1_BAJO,501,25.0000
1,Q2_MEDIO_BAJO,501,25.0000
2,Q3_MEDIO_ALTO,501,25.0000
3,Q4_ALTO,501,25.0000



=== TDNC POR CUARTIL ===


,count,mean,std,min,25%,50%,75%,max
TARGET_B,,,,,,,,
Q1_BAJO,501.0000,0.0063,0.0077,0.0000,0.0000,0.0022,0.0115,0.0249
Q2_MEDIO_BAJO,501.0000,0.0571,0.0197,0.0251,0.0401,0.0560,0.0745,0.0922
Q3_MEDIO_ALTO,501.0000,0.1412,0.0332,0.0926,0.1115,0.1377,0.1678,0.2115
Q4_ALTO,501.0000,0.4353,0.2105,0.2115,0.2806,0.3676,0.5353,1.1363


In [5]:
from sklearn.model_selection import train_test_split

def split_target_b(df_asesores, test_size=0.20, random_state=42):
    '''
    Split estratificado por asesor para Target_B.
    Ningún asesor puede aparecer en train Y test.
    '''
    EXCLUIR_B = [
        'TARGET_B', 'TDNC', 'DIAS_MORA',
        'PRIMERA_OBS', 'ULTIMA_OBS',
        'ASESOR', 'NOMBRE', 'NUMERO_EMPLEADO',
        'PASE1', 'PASE15', 'PASE30',
        'DIAS_MORA_RANGO',
    ]
    feature_cols = [c for c in df_asesores.columns if c not in EXCLUIR_B]

    df_clean = df_asesores.dropna(subset=['TARGET_B'])
    n_dropped = len(df_asesores) - len(df_clean)
    if n_dropped > 0:
        print(f'Asesores sin TARGET_B excluidos: {n_dropped}')

    X = df_clean[feature_cols]
    y = df_clean['TARGET_B'].astype(str)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    # Verificacion: ningun asesor en ambos conjuntos
    assert len(set(X_train.index) & set(X_test.index)) == 0
    print('Anti-leakage OK: ningun asesor en train Y test simultáneamente')

    return X_train, X_test, y_train, y_test, feature_cols


print('=== TARGET_B — Split por Asesor ===')
X_train_B, X_test_B, y_train_B, y_test_B, features_B = split_target_b(df_asesores)

print(f'\nTrain: {len(X_train_B):,} asesores ({len(X_train_B)/len(df_asesores)*100:.1f}%)')
print(f'Test : {len(X_test_B):,} asesores ({len(X_test_B)/len(df_asesores)*100:.1f}%)')
print(f'Features disponibles: {len(features_B)}')
print('\nDistribucion train:')
print(y_train_B.value_counts().to_string())
print('\nDistribucion test:')
print(y_test_B.value_counts().to_string())


=== TARGET_B — Split por Asesor ===
Anti-leakage OK: ningun asesor en train Y test simultáneamente

Train: 1,603 asesores (80.0%)
Test : 401 asesores (20.0%)
Features disponibles: 35

Distribucion train:
TARGET_B
Q2_MEDIO_BAJO    401
Q1_BAJO          401
Q4_ALTO          401
Q3_MEDIO_ALTO    400

Distribucion test:
TARGET_B
Q3_MEDIO_ALTO    101
Q4_ALTO          100
Q1_BAJO          100
Q2_MEDIO_BAJO    100


## 4. Decisión formal de target principal

Criterio objetivo: si alguna clase tiene menos de 30 observaciones en train,
Target_B es inviable y se usa Target_A como principal.


In [6]:
print('=== VERIFICACIÓN DE VIABILIDAD — TARGET_B ===')

min_clase_train = y_train_B.value_counts().min()
min_clase_test  = y_test_B.value_counts().min()

print(f'Clase minima en train : {min_clase_train} observaciones')
print(f'Clase minima en test  : {min_clase_test} observaciones')

if min_clase_train < 30:
    decision = 'TARGET_A'
    razon = 'Target_B inviable: clase con menos de 30 observaciones en train'
elif min_clase_test < 15:
    decision = 'TARGET_B con k-fold CV'
    razon = 'Target_B viable pero clases pequeñas en test: usar k-fold estratificado'
else:
    decision = 'TARGET_B'
    razon = 'Target_B viable: todas las clases tienen observaciones suficientes'

print(f'\n' + '='*55)
print(f'  DECISION FORMAL: {decision}')
print(f'  RAZON: {razon}')
print('='*55)


=== VERIFICACIÓN DE VIABILIDAD — TARGET_B ===
Clase minima en train : 400 observaciones
Clase minima en test  : 100 observaciones

  DECISION FORMAL: TARGET_B
  RAZON: Target_B viable: todas las clases tienen observaciones suficientes


## 5. Guardar splits en data/interim/

In [7]:
# Target_A
df_train_A = X_train_A.copy()
df_train_A['DIAS_MORA_RANGO'] = y_train_A.values
df_train_A['HOY'] = df.loc[y_train_A.index, 'HOY'].values

df_test_A = X_test_A.copy()
df_test_A['DIAS_MORA_RANGO'] = y_test_A.values
df_test_A['HOY'] = df.loc[y_test_A.index, 'HOY'].values

df_train_A.to_parquet(INTERIM / 'target_a_train.parquet', index=False)
df_test_A.to_parquet(INTERIM  / 'target_a_test.parquet',  index=False)
print(f'target_a_train.parquet — {len(df_train_A):,} filas')
print(f'target_a_test.parquet  — {len(df_test_A):,} filas')

# Target_B
df_train_B = X_train_B.copy()
df_train_B['TARGET_B'] = y_train_B.values
df_train_B['TDNC']     = df_asesores.loc[X_train_B.index, 'TDNC'].values

df_test_B = X_test_B.copy()
df_test_B['TARGET_B'] = y_test_B.values
df_test_B['TDNC']     = df_asesores.loc[X_test_B.index, 'TDNC'].values

df_train_B.to_parquet(INTERIM / 'target_b_train.parquet', index=False)
df_test_B.to_parquet(INTERIM  / 'target_b_test.parquet',  index=False)
df_asesores.to_parquet(INTERIM / 'target_b_full.parquet', index=False)
print(f'target_b_train.parquet — {len(df_train_B):,} asesores')
print(f'target_b_test.parquet  — {len(df_test_B):,} asesores')
print(f'target_b_full.parquet  — {len(df_asesores):,} asesores')
print('\nTodos los splits guardados en data/interim/')


target_a_train.parquet — 993,911 filas
target_a_test.parquet  — 224,920 filas
target_b_train.parquet — 1,603 asesores
target_b_test.parquet  — 401 asesores
target_b_full.parquet  — 2,004 asesores

Todos los splits guardados en data/interim/
